# Data Engineering Class (T13001C)
## Introduction to pandas
Professor: Eduardo de Avila-Armenta | eduardo.deavila@tec.mx | github repo: https://github.com/LalodeAvila369/DataEng-class-TI3001C


## Class Summary

This notebook collects **every piece of code shown in the Week 1 Thursday class** (Topic 2.1 — Introduction to
pandas), plus the live activities we ran, in one runnable place. Use it to follow along, re-run any example, or
review before the lab.

**What we covered today:**

1. **Background — data & pandas.** What real data actually looks like (structured / semi-structured /
   unstructured), and how pandas builds on plain Python lists and NumPy arrays by adding *labels*.
2. **2.1.1 — Series & DataFrames.** The two core pandas objects: a `Series` (one labeled column) and a
   `DataFrame` (a table made of Series that share an index).
3. **2.1.2 — Dates & Time.** Why raw dates are just text until you call `pd.to_datetime()`, the most useful
   `.dt` accessors, and how to deal with messy/mixed date formats.
4. **2.1.3 — Indexing, Filtering & Iteration.** `.loc` (by label) vs. `.iloc` (by position), boolean filtering
   (`&` / `|` / `~`), and why row-by-row loops (`.iterrows()`) should almost always be replaced by vectorized
   operations.
5. **2.1.4 — Fundamental Operations.** A first look at four families of operations — indexing & selection
   (already covered), data cleaning & missing values (deep dive in Week 3), computation & statistics (today's
   focus — `.describe()`, `.info()`, `.value_counts()`, etc.), and manipulation & structuring (deep dive in
   Week 3 — sorting, grouping, combining tables).
6. **2.1.5 — Derived Columns.** Building new columns from columns you already have — arithmetic, boolean
   flags, and binned categories (`pd.cut()`) — plus why vectorized math beats `.apply()`.

**Activities we ran:**

- **The 500-Row Race** — a live, timed comparison of Excel vs. pandas (with and without AI assistance).
- **Build Your Own Dates Dataset** — a personal mini-activity converting real dates with `pd.to_datetime()`.
- **Practical Activity: Simulated Diabetes Patient Dataset** — a full feature-engineering exercise (BMI,
  age bins, clinical HbA1c categories, splitting a messy `"systolic/diastolic"` string, and a composite
  `risk_factor_count` that turned out to genuinely predict the outcome).
- **Recap Quiz (Kahoot!)** — a fast, competitive review of everything above (question bank kept separately).

**Then, in the hands-on lab block**, we applied everything to (a synthetic stand-in for) the real running
project dataset — Olist, a Brazilian e-commerce dataset — across 5 exercises: load & inspect, dates, select &
filter, describe, and derive.

Every code cell below is commented, and every cell is preceded by a short explanation of *why* we're doing it,
not just *what* it does.


## Setup

We only need two libraries all class: **pandas** (our main tool) and **NumPy** (pandas is built on top of it,
and we'll use it directly a couple of times for fast math and random numbers).


In [35]:
# pandas: the library this entire class is about — tables (DataFrames) and labeled columns (Series)
import pandas as pd

# numpy: pandas is built on top of numpy; we use it directly for vectorized math and random numbers
import numpy as np

# so numbers print without excessive scientific notation / long decimal tails in this notebook
pd.set_option("display.precision", 3)

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)


pandas version: 2.2.3
numpy version: 1.26.4


## Background — Data & pandas

### What does real data look like?

Real-world data shows up in three broad shapes: **structured** (fixed rows & columns, like a CSV),
**semi-structured** (some organization, but no fixed schema, like JSON), and **unstructured** (no
predefined structure at all, like free text). pandas is built for the first one — but real Data Engineering
pipelines have to deal with all three.


In [36]:
import io

# --- Structured: a CSV — rows and columns, fixed schema, ready for pandas immediately ---
csv_text = """order_id,customer,price
o1,Ana,29.90
o2,Luis,15.50
o3,Marta,89.00
"""
# io.StringIO lets us treat the text above as if it were a file, so pd.read_csv can read it directly
structured_df = pd.read_csv(io.StringIO(csv_text))
print("STRUCTURED (csv) -> ready to use immediately:")
print(structured_df)

# --- Semi-structured: JSON — some organization (keys/values, nesting), but no fixed table schema ---
import json
json_text = """{
  "order_id": "o1",
  "customer": "Ana",
  "items": [
    {"sku": "A1", "price": 29.90}
  ]
}"""
order_dict = json.loads(json_text)
# json_normalize "flattens" nested JSON into a table -- this is the parsing step semi-structured data needs
# before pandas can work with it directly
semi_structured_df = pd.json_normalize(order_dict, record_path="items", meta=["order_id", "customer"])
print("\nSEMI-STRUCTURED (json) -> needed a parsing/flattening step first:")
print(semi_structured_df)

# --- Unstructured: free text — no predefined structure; pandas can't compute on this directly ---
review_text = (
    "Hi, my order #o1 arrived on 05/14 but the color was wrong. "
    "Contact me at ana@email.com or 555-2938. Disappointed."
)
print("\nUNSTRUCTURED (free text) -> can't compute on it directly; would need NLP/regex to extract fields:")
print(review_text)


STRUCTURED (csv) -> ready to use immediately:
  order_id customer  price
0       o1      Ana   29.9
1       o2     Luis   15.5
2       o3    Marta   89.0

SEMI-STRUCTURED (json) -> needed a parsing/flattening step first:
  sku  price order_id customer
0  A1   29.9       o1      Ana

UNSTRUCTURED (free text) -> can't compute on it directly; would need NLP/regex to extract fields:
Hi, my order #o1 arrived on 05/14 but the color was wrong. Contact me at ana@email.com or 555-2938. Disappointed.


**Takeaway:** the less structured the data, the more work is needed before you can analyze it. Structured
data loads straight into pandas; semi-structured data needs parsing/flattening; unstructured data needs
feature extraction (NLP, computer vision, OCR) before it becomes usable at all.

### From lists to tables: where pandas fits

pandas doesn't come out of nowhere — it's the same idea as a plain Python list or a NumPy array, with
**labels** added on top (row and column names). This one idea is the foundation for everything else we do
today.


In [37]:
# 1) A plain Python list: ordered values, but no labels attached
prices_list = [29.90, 15.50, 89.00]
print("Python list:      ", prices_list)

# 2) A NumPy array: same idea, but built for fast, vectorized math -- still no labels
prices_array = np.array([29.90, 15.50, 89.00])
print("NumPy array:       ", prices_array, "  (x2 ->", prices_array * 2, ")")

# 3) A matrix: just a 2D NumPy array -- rows and columns of numbers, still no labels
price_matrix = np.array([[29.90, 7.50], [15.50, 5.00]])
print("NumPy matrix:\n", price_matrix)

# 4) A pandas Series: a NumPy array PLUS labels (here, a name and a row index)
prices_series = pd.Series(prices_list, name="price")
print("\npandas Series (array + labels):")
print(prices_series)


Python list:       [29.9, 15.5, 89.0]
NumPy array:        [29.9 15.5 89. ]   (x2 -> [ 59.8  31.  178. ] )
NumPy matrix:
 [[29.9  7.5]
 [15.5  5. ]]

pandas Series (array + labels):
0    29.9
1    15.5
2    89.0
Name: price, dtype: float64


**pandas = arrays and matrices, plus labels and convenience.** That's the whole idea — everything else we
learn today builds on it.

### What is pandas, and why not just use Excel?

pandas is an open-source Python library for working with tabular data, built on top of NumPy. Compared to
Excel/Sheets, pandas scales to millions of rows, is fully reproducible (it's code, not manual clicks), works
natively with version control (git), and handles complex transformations that are painful in spreadsheet
formulas. Excel is still great for a quick look or sharing with non-technical people — reach for pandas once
the work needs to be *repeated*, *automated*, or *trusted*. (Alternatives for later in the course: **Polars**,
**SQL**, and **Spark/Dask** for data too big to fit in memory.)


## Live Activity — The 500-Row Race

In class, four "teams" raced to build the same table by hand: **Professor** (pandas only), **Team Excel**
(spreadsheet formulas only), **Team AI Excel** (spreadsheet + an AI assistant), and **Team AI pandas**
(Python + pandas + an AI assistant). The task was identical for everyone:

- **Column A:** 500 random integers (1–1000)
- **Column B:** 500 random floats (0–100)
- **Column C:** $\sqrt{A^2 + B^2}$
- **Column D:** $(A + B - C) / \pi$

Below is the pandas solution — the whole point of the activity is to *feel* how much faster and less
error-prone this is than doing the same thing by hand in a spreadsheet.


In [38]:
import time

# a fixed seed makes this reproducible -- everyone (including you, re-running this) gets the same numbers
rng = np.random.default_rng(1)
n = 500

start = time.perf_counter()

race = pd.DataFrame({
    "A": rng.integers(1, 1001, size=n),   # 500 random integers from 1 to 1000
    "B": rng.uniform(0, 100, size=n),     # 500 random floats from 0 to 100
})
race["C"] = np.sqrt(race["A"] ** 2 + race["B"] ** 2)          # Euclidean-style combination of A and B
race["D"] = (race["A"] + race["B"] - race["C"]) / np.pi        # final derived column

elapsed_ms = (time.perf_counter() - start) * 1000

print(race.head())
print(f"\nBuilt and computed all 500 rows in {elapsed_ms:.2f} ms.")


     A       B        C       D
0  474  60.651  477.865  18.076
1  512   3.405  512.011   1.080
2  756  42.946  757.219  13.282
3  951  68.520  953.465  21.026
4   35  15.635   38.333   3.916

Built and computed all 500 rows in 15.99 ms.


**Debrief:** the pandas version above ran in a handful of milliseconds, start to finish, for all 500 rows —
no dragging fill handles, no fixing a wrong cell reference, no risk of one row using a different formula by
accident. That gap only gets wider once "500 rows" becomes 5 million, which is the world this course is
actually preparing you for.


## 2.1.1 — Series & DataFrames

A **Series** is a single labeled column of data (values + an index) — think of it as one column of a
spreadsheet with row labels attached. A **DataFrame** is a table: multiple Series sharing the same index.
Every column in a DataFrame is a Series. This is the object you'll spend most of this course inside of.

From here on, we build one running example — an `orders` table — and keep adding to it as the class
progresses, exactly like we did on the slides.


In [39]:
# A Series: one labeled column. "name" becomes the column's label if it ever joins a DataFrame.
prices = pd.Series([29.90, 15.50, 89.00], name="price")
print(prices)


0    29.9
1    15.5
2    89.0
Name: price, dtype: float64


In [40]:
# A DataFrame: our running example for the rest of the class.
# Every column here is technically a Series -- they just happen to share the same index (0, 1, 2).
orders = pd.DataFrame({
    "order_id":       ["o1", "o2", "o3"],
    "price":          [29.90, 15.50, 89.00],
    "freight_value":  [7.50, 5.00, 12.30],
    # dates start out as plain TEXT -- we'll convert this properly in the next section
    "order_purchase_timestamp": [
        "2018-05-14 10:23:00", "2018-05-16 08:05:00", "2018-05-20 19:47:00",
    ],
})
orders.head()


,order_id,price,freight_value,order_purchase_timestamp
0,o1,29.9,7.5,2018-05-14 10:23:00
1,o2,15.5,5.0,2018-05-16 08:05:00
2,o3,89.0,12.3,2018-05-20 19:47:00


In [41]:
# .shape tells you (rows, columns) at a glance -- a quick sanity check after loading/building any DataFrame
orders.shape


(3, 4)

**Visually:** picture the whole `orders` table as a grid. Any *one column* of that grid (say, just `price`)
is a Series on its own. The whole grid together is the DataFrame.


## 2.1.2 — Dates & Time

Dates almost always arrive as **plain text** (like `"2018-05-14 10:23:00"` above) — not usable as dates
until you convert them. Once converted with `pd.to_datetime()`, pandas unlocks year, month, weekday, time
differences, and more via the `.dt` accessor. Skipping this step is exactly the kind of thing an
AI-generated pipeline tends to get wrong.


In [42]:
# Convert the text column into real datetime objects (this OVERWRITES the column in place)
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

# Now that it's a real datetime column, .dt unlocks things like the weekday name
orders["purchase_dow"] = orders["order_purchase_timestamp"].dt.day_name()

orders[["order_purchase_timestamp", "purchase_dow"]]


,order_purchase_timestamp,purchase_dow
0,2018-05-14 10:23:00,Monday
1,2018-05-16 08:05:00,Wednesday
2,2018-05-20 19:47:00,Sunday


### Cheat sheet: `pd.to_datetime` & `.dt`

A quick reference for parsing dates and pulling information back out of them.


In [43]:
# --- Parsing text into dates: the options you'll actually use ---

pd.to_datetime(orders["order_purchase_timestamp"])
# auto-detects the format (already converted above, so this just confirms it still works)

pd.to_datetime(["2018-05-14"], format="%Y-%m-%d")
# an EXACT format string: faster and safer than auto-detection when you know the format

pd.to_datetime(["14/05/2018"], dayfirst=True)
# tells pandas to read DD/MM/YYYY instead of the US-style MM/DD/YYYY default

pd.to_datetime(["2018-05-14", "14/05/2018"], format="mixed", dayfirst=True)
# use this when DIFFERENT ROWS use different formats -- pandas guesses each one individually

pd.to_datetime(["not a date"], errors="coerce")
# turns anything unparseable into NaT (pandas' "missing date") instead of crashing your script


DatetimeIndex(['NaT'], dtype='datetime64[ns]', freq=None)

In [44]:
# --- The .dt accessor: pulling information back out of a real datetime column ---
ts = orders["order_purchase_timestamp"]

print("year:        ", ts.dt.year.tolist())
print("month:       ", ts.dt.month.tolist())
print("day:         ", ts.dt.day.tolist())
print("hour:        ", ts.dt.hour.tolist())
print("minute:      ", ts.dt.minute.tolist())
print("day_name():  ", ts.dt.day_name().tolist())
print("month_name():", ts.dt.month_name().tolist())
print("dayofweek:   ", ts.dt.dayofweek.tolist(), " (0=Monday ... 6=Sunday)")
print("quarter:     ", ts.dt.quarter.tolist())
print("date only:   ", ts.dt.date.tolist())


year:         [2018, 2018, 2018]
month:        [5, 5, 5]
day:          [14, 16, 20]
hour:         [10, 8, 19]
minute:       [23, 5, 47]
day_name():   ['Monday', 'Wednesday', 'Sunday']
month_name(): ['May', 'May', 'May']
dayofweek:    [0, 2, 6]  (0=Monday ... 6=Sunday)
quarter:      [2, 2, 2]
date only:    [datetime.date(2018, 5, 14), datetime.date(2018, 5, 16), datetime.date(2018, 5, 20)]


### Standardizing a messy CSV column

A very common real-world problem: a CSV column where different rows use *different* date formats. The
fix is always the same two-step recipe: **parse once** with `format="mixed"`, then **render however you
need** with `.dt.strftime()`.


In [45]:
# A column with three genuinely different formats for the same kind of date
messy = pd.DataFrame({"raw_date": ["2026-08-26", "14/05/2026", "Aug 1, 2026"]})

# Step 1: parse everything into real datetimes, letting pandas infer each row's format individually
parsed = pd.to_datetime(messy["raw_date"], format="mixed", dayfirst=True)

# Step 2: render every value in ONE consistent output format, regardless of how it started out
messy["clean_date"] = parsed.dt.strftime("%d/%m/%Y")

messy


,raw_date,clean_date
0,2026-08-26,26/08/2026
1,14/05/2026,14/05/2026
2,"Aug 1, 2026",01/08/2026


### Quick Activity — Build Your Own Dates Dataset

In class, each pair built a small 5-row DataFrame of their own — either (a) important dates in their life,
or (b) tomorrow's schedule — then converted the date column and pulled something out of it. Here's the
blank template we used, followed by one worked example (a wedding weekend) so you have a complete
reference solution.


In [46]:
# The blank template handed out in class (fill in your own events/dates to try this yourself)
my_dates = pd.DataFrame({
    "event": ["___", "___", "___", "___", "___"],
    "date":  ["___", "___", "___", "___", "___"],
})
# my_dates["date"] = pd.to_datetime(my_dates["date"])
# then try: .dt.day_name(), .dt.month, or a time difference between two rows
my_dates


,event,date
0,___,___
1,___,___
2,___,___
3,___,___
4,___,___


In [47]:
# Worked example: a wedding weekend, as a sequence of events across several days
wedding = pd.DataFrame({
    "event": ["Civil Wedding", "Ceremony", "After-wedding Brunch", "Honeymoon", "Return to Work"],
    "date":  ["2026-11-06", "2026-11-07", "2026-11-08", "2026-11-09", "2026-11-23"],
})
wedding["date"] = pd.to_datetime(wedding["date"])

# Extract the weekday name for each event
wedding["day_name"] = wedding["date"].dt.day_name()

# Derived column: how many days after the Civil Wedding did each event happen?
wedding["days_since_civil"] = (wedding["date"] - wedding["date"].iloc[0]).dt.days

wedding


,event,date,day_name,days_since_civil
0,Civil Wedding,2026-11-06,Friday,0
1,Ceremony,2026-11-07,Saturday,1
2,After-wedding Brunch,2026-11-08,Sunday,2
3,Honeymoon,2026-11-09,Monday,3
4,Return to Work,2026-11-23,Monday,17


**Debrief:** in class, a few pairs hit surprises — a typo pandas rejected outright, an ambiguous format
like `05/03` (May 3rd, or March 5th?), or a missing year. This "small" conversion step is exactly what
breaks real pipelines when it's skipped.


## 2.1.3 — Indexing, Filtering & Iteration

Two ways to select data: **`.loc[]`** selects by *label* (row/column name), **`.iloc[]`** selects by
*position* (row/column number). Confusing these two is one of the most common pandas bugs. With the
default `0, 1, 2` index, the label and the position happen to look the same — which is exactly why the
distinction is easy to miss until it bites you on a DataFrame with a different index.


In [48]:
# .loc: by LABEL. Row label 1, column label "price".
orders.loc[1, "price"]


15.5

In [49]:
# .iloc: by POSITION. Rows at positions 0 and 1 (position 2 is excluded, like normal Python slicing).
orders.iloc[0:2]


,order_id,price,freight_value,order_purchase_timestamp,purchase_dow
0,o1,29.9,7.5,2018-05-14 10:23:00,Monday
1,o2,15.5,5.0,2018-05-16 08:05:00,Wednesday


**Quick Check (predict before you run):** do `orders.iloc[2, 0]` and `orders.loc[2, "order_id"]` give the
same result here? Why? What would break this if the index weren't `0, 1, 2`?


In [50]:
# With the default integer index, position 2 and label 2 happen to coincide -- but that's a coincidence
# of THIS index, not a general rule. Change the index and only .loc would still point at the same row.
print("orders.iloc[2, 0]:      ", orders.iloc[2, 0])
print("orders.loc[2, 'order_id']:", orders.loc[2, "order_id"])


orders.iloc[2, 0]:       o3
orders.loc[2, 'order_id']: o3


### Filtering (boolean masks)

Boolean filtering is how you'll do almost all row selection in this course: write a condition, get back a
column of `True`/`False`, and pandas keeps the `True` rows.


In [51]:
# A condition like this creates a column of True/False -- then orders[...] keeps only the True rows
expensive = orders[orders["price"] > 20]
expensive


,order_id,price,freight_value,order_purchase_timestamp,purchase_dow
0,o1,29.9,7.5,2018-05-14 10:23:00,Monday
2,o3,89.0,12.3,2018-05-20 19:47:00,Sunday


### Combining filters: AND / OR / NOT

Each condition needs its **own parentheses** — `(a) & (b)`, never `a & b` — and you use `&` / `|` / `~`
instead of Python's `and` / `or` / `not` (which don't work element-by-element on a Series).


In [52]:
# AND: & -- both conditions must be true
filtered = orders[
    (orders["price"] > 20) & (orders["freight_value"] < 10)
]
print("AND (price > 20 & freight < 10):")
print(filtered)

# OR: | -- either condition can be true
print("\nOR (price > 80 | price < 16):")
print(orders[(orders["price"] > 80) | (orders["price"] < 16)])

# NOT: ~ -- flips the condition
print("\nNOT (~(price > 20)):")
print(orders[~(orders["price"] > 20)])


AND (price > 20 & freight < 10):
  order_id  price  freight_value order_purchase_timestamp purchase_dow
0       o1   29.9            7.5      2018-05-14 10:23:00       Monday

OR (price > 80 | price < 16):
  order_id  price  freight_value order_purchase_timestamp purchase_dow
1       o2   15.5            5.0      2018-05-16 08:05:00    Wednesday
2       o3   89.0           12.3      2018-05-20 19:47:00       Sunday

NOT (~(price > 20)):
  order_id  price  freight_value order_purchase_timestamp purchase_dow
1       o2   15.5            5.0      2018-05-16 08:05:00    Wednesday


### Filtering by list, text & dates

The same "make a True/False column, keep the True rows" idea applies everywhere — lists (`.isin()`), text
(`.str.contains()`), and dates (plain comparison operators, once the column is a real datetime).


In [53]:
# Filtering by a list of values
print("isin (order_id is o1 or o3):")
print(orders[orders["order_id"].isin(["o1", "o3"])])

# Filtering by text content
print("\nstr.contains (order_id contains '2'):")
print(orders[orders["order_id"].str.contains("2")])

# Filtering by date (works because order_purchase_timestamp is a real datetime column now)
print("\ndate filter (purchased after 2018-05-15):")
print(orders[orders["order_purchase_timestamp"] > "2018-05-15"])


isin (order_id is o1 or o3):
  order_id  price  freight_value order_purchase_timestamp purchase_dow
0       o1   29.9            7.5      2018-05-14 10:23:00       Monday
2       o3   89.0           12.3      2018-05-20 19:47:00       Sunday

str.contains (order_id contains '2'):
  order_id  price  freight_value order_purchase_timestamp purchase_dow
1       o2   15.5            5.0      2018-05-16 08:05:00    Wednesday

date filter (purchased after 2018-05-15):
  order_id  price  freight_value order_purchase_timestamp purchase_dow
1       o2   15.5            5.0      2018-05-16 08:05:00    Wednesday
2       o3   89.0           12.3      2018-05-20 19:47:00       Sunday


### Cheat sheet: filtering & boolean masks

A combined, real example: expensive orders that also arrived after May 15th.


In [54]:
# Comparison filters:
#   df[df["col"] > x]              df[df["col"] == x]            df[df["col"] != x]
#   df[df["col"].between(a, b)]    df[df["col"].isna()]          df[df["col"].notna()]
#
# Combining, lists & text:
#   df[(cond1) & (cond2)]   # AND         df[(cond1) | (cond2)]   # OR
#   df[~(cond1)]            # NOT         df[df["col"].isin([...])]
#   df[df["col"].str.contains("x")]

# Real example: expensive AND arrived after May 15
orders[
    (orders["price"] > 20) &
    (orders["order_purchase_timestamp"] > "2018-05-15")
]


,order_id,price,freight_value,order_purchase_timestamp,purchase_dow
2,o3,89.0,12.3,2018-05-20 19:47:00,Sunday


### Why avoid iteration (when you can)?

A Python `for` loop touches your data **one row at a time** — exactly what slowed down Team Excel and Team
AI Excel in the 500-Row Race. Vectorized pandas operations run in optimized C under the hood, on the
**whole column at once**. Let's actually measure the difference on 10,000 rows.


In [55]:
# Benchmark: .iterrows() vs .itertuples() vs a fully vectorized column operation
bench = pd.DataFrame({"a": np.random.rand(10_000), "b": np.random.rand(10_000)})

t0 = time.perf_counter()
total = 0
for idx, row in bench.iterrows():
    total += row["a"] + row["b"]
t_iterrows = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
total = 0
for row in bench.itertuples():
    total += row.a + row.b
t_itertuples = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
result = bench["a"] + bench["b"]
t_vectorized = (time.perf_counter() - t0) * 1000

print(f".iterrows():   {t_iterrows:8.2f} ms")
print(f".itertuples(): {t_itertuples:8.2f} ms")
print(f"vectorized:    {t_vectorized:8.2f} ms")
print(f"\n.iterrows() was about {t_iterrows / t_vectorized:.0f}x slower than the vectorized version.")


.iterrows():     114.41 ms
.itertuples():     5.13 ms
vectorized:        0.92 ms

.iterrows() was about 124x slower than the vectorized version.


**Rule of thumb:** if it can be written as a column operation, write it that way. Save loops for genuinely
row-specific logic (like calling an external API once per row) — everything else in this course: vectorize.


In [56]:
# .iterrows() gives you a full Series per row -- flexible, but the slowest option
for idx, row in orders.iterrows():
    print(idx, row["order_id"], row["price"])


0 o1 29.9
1 o2 15.5
2 o3 89.0


In [57]:
# .itertuples() gives a lightweight named tuple per row -- much faster than .iterrows() if you must loop
for row in orders.itertuples():
    print(row.Index, row.order_id, row.price)


0 o1 29.9
1 o2 15.5
2 o3 89.0


## 2.1.4 — Fundamental Operations

"Fundamental operations" splits into four categories. We already covered the first one; today's proper
focus is the third one; the other two get a first look now and a deep dive in **Week 3**:

| Category | When |
|---|---|
| **Indexing & Selection** — finding the rows/columns you need | Just did this (2.1.3) |
| **Data Cleaning & Missing Values** — handling incomplete/incorrect data | Deep dive: Week 3 |
| **Computation & Statistics** — summarizing what your data says | **Today's focus** |
| **Manipulation & Structuring** — sorting, grouping, combining tables | Deep dive: Week 3 |


### Data cleaning & missing values (first look)

Real data always has holes — missing prices, blank dates, duplicate rows. Pandas marks missing values as
`NaN` (numbers) or `NaT` (dates) — not the same as `0` or an empty string. We clean, validate and impute
properly in Week 3; for now, here's what the key methods do.


In [58]:
# A small copy of orders with a missing value and a duplicate row, just to demonstrate the methods below
messy_orders = orders.copy()
messy_orders.loc[1, "price"] = np.nan                      # simulate a missing price
messy_orders = pd.concat([messy_orders, messy_orders.iloc[[0]]], ignore_index=True)  # simulate a duplicate row

print("isna() flags missing values:")
print(messy_orders["price"].isna().tolist())

print("\ndropna() removes rows with any missing value in the given subset:")
print(messy_orders.dropna(subset=["price"]).shape, "  (started with", messy_orders.shape, ")")

print("\nfillna(0) replaces missing values:")
print(messy_orders["price"].fillna(0).tolist())

print("\nduplicated() flags duplicate rows (by order_id here):")
print(messy_orders["order_id"].duplicated().tolist())

print("\ndrop_duplicates() removes them:")
print(messy_orders.drop_duplicates(subset=["order_id"]).shape)


isna() flags missing values:
[False, True, False, False]

dropna() removes rows with any missing value in the given subset:
(3, 5)   (started with (4, 5) )

fillna(0) replaces missing values:
[29.9, 0.0, 89.0, 29.9]

duplicated() flags duplicate rows (by order_id here):
[False, False, False, True]

drop_duplicates() removes them:
(3, 5)


### Computation & statistics

Before transforming anything, **look at it** — this is the EDA habit from Monday's discussion.
`.describe()`, `.info()`, `.value_counts()` answer most first questions.


In [59]:
# .describe() -- count, mean, std, min, quartiles, max, all in one call
orders["price"].describe()


count     3.00
mean     44.80
std      38.95
min      15.50
25%      22.70
50%      29.90
75%      59.45
max      89.00
Name: price, dtype: float64

In [60]:
# Individual statistics, when you only need one number
orders["price"].mean(), orders["price"].sum()


(44.800000000000004, 134.4)

In [61]:
# .info() -- column names, dtypes, and non-null counts, all at once (useful right after loading any file)
orders.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   order_id                  3 non-null      object        
 1   price                     3 non-null      float64       
 2   freight_value             3 non-null      float64       
 3   order_purchase_timestamp  3 non-null      datetime64[ns]
 4   purchase_dow              3 non-null      object        
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 252.0+ bytes


### Cheat sheet: computation & statistics

```python
df.describe()                          # count, mean, std, min, quartiles, max
df.info()                              # column names, dtypes, non-null counts
df["col"].mean() / .sum()              # average / total
df["col"].min() / .max()               # smallest / largest value
df["col"].std()                        # spread (standard deviation)
df["col"].unique() / .nunique()        # distinct values / how many
df["col"].value_counts()               # frequency of each value
```


In [62]:
# A couple of these in action on our purchase_dow column
print("unique:  ", orders["purchase_dow"].unique().tolist())
print("nunique: ", orders["purchase_dow"].nunique())
print("value_counts:")
print(orders["purchase_dow"].value_counts())


unique:   ['Monday', 'Wednesday', 'Sunday']
nunique:  3
value_counts:
purchase_dow
Monday       1
Wednesday    1
Sunday       1
Name: count, dtype: int64


### Manipulation & structuring (first look)

Reordering rows, grouping by category, and combining tables from different sources. This is where a lot of
real Data Engineering work happens — most raw data lives in more than one table. We build these skills
properly in Week 3; here's a first look at each.


In [63]:
# sort_values: reorder rows by a column
print("sorted by price (descending):")
print(orders.sort_values("price", ascending=False)[["order_id", "price"]])

# groupby: aggregate by category (here: average price per weekday)
print("\naverage price per weekday:")
print(orders.groupby("purchase_dow")["price"].mean())

# concat: stack tables on top of each other (e.g. adding a new batch of orders)
more_orders = pd.DataFrame({
    "order_id": ["o4"], "price": [50.0], "freight_value": [9.0],
    "order_purchase_timestamp": pd.to_datetime(["2018-06-01"]),
    "purchase_dow": ["Friday"],
})
print("\nconcat (stacking a new order onto the table):")
print(pd.concat([orders, more_orders], ignore_index=True))

# merge: join tables side by side, by a shared key (e.g. adding seller names from another table)
sellers = pd.DataFrame({"order_id": ["o1", "o2", "o3"], "seller": ["Ana Shop", "Luis Store", "Marta Goods"]})
print("\nmerge (joining in seller names by order_id):")
print(orders.merge(sellers, on="order_id")[["order_id", "price", "seller"]])


sorted by price (descending):
  order_id  price
2       o3   89.0
0       o1   29.9
1       o2   15.5

average price per weekday:
purchase_dow
Monday       29.9
Sunday       89.0
Wednesday    15.5
Name: price, dtype: float64

concat (stacking a new order onto the table):
  order_id  price  freight_value order_purchase_timestamp purchase_dow
0       o1   29.9            7.5      2018-05-14 10:23:00       Monday
1       o2   15.5            5.0      2018-05-16 08:05:00    Wednesday
2       o3   89.0           12.3      2018-05-20 19:47:00       Sunday
3       o4   50.0            9.0      2018-06-01 00:00:00       Friday

merge (joining in seller names by order_id):
  order_id  price       seller
0       o1   29.9     Ana Shop
1       o2   15.5   Luis Store
2       o3   89.0  Marta Goods


## 2.1.5 — Derived Columns

A **derived column** is built from columns you already have — this is where raw data starts turning into
something useful for analysis. Prefer vectorized math over `.apply()` when possible: same result, much
faster.


In [64]:
# The simplest derived column: adding two existing columns together
orders["total_value"] = orders["price"] + orders["freight_value"]
orders[["order_id", "total_value"]]


,order_id,total_value
0,o1,37.4
1,o2,20.5
2,o3,101.3


### More derived columns: flags & categories

A derived column doesn't have to be a number — booleans and categories are features too, and this is
exactly the kind of thing a Machine Learning model eats: a boolean flag, or a binned category.


In [65]:
# A boolean (True/False) derived column
orders["is_expensive"] = orders["price"] > 20
print(orders[["order_id", "is_expensive"]])

# A categorical derived column: binning a continuous value into labeled buckets
orders["price_tier"] = pd.cut(
    orders["price"], bins=[0, 20, 50, 200],
    labels=["low", "medium", "high"]
)
print("\n", orders[["order_id", "price_tier"]])


  order_id  is_expensive
0       o1          True
1       o2         False
2       o3          True

   order_id price_tier
0       o1     medium
1       o2        low
2       o3       high


### Vectorized vs. `.apply()`

`.apply(..., axis=1)` runs your function **once per row** — same idea as the loops from 2.1.3, just hidden
inside a method call. Use `.apply()` only when the logic genuinely can't be written as column math.


In [66]:
# Same result as orders["total_value"] above, but computed row-by-row with .apply() instead of vectorized math
total_value_via_apply = orders.apply(
    lambda row: row["price"] + row["freight_value"], axis=1
)
print("Same result as vectorized?", (total_value_via_apply == orders["total_value"]).all())

# Benchmark on a bigger table to see WHY vectorized math is preferred
big = pd.DataFrame({"price": np.random.rand(10_000) * 100, "freight_value": np.random.rand(10_000) * 20})

t0 = time.perf_counter()
_ = big.apply(lambda row: row["price"] + row["freight_value"], axis=1)
t_apply = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
_ = big["price"] + big["freight_value"]
t_vectorized = (time.perf_counter() - t0) * 1000

print(f".apply(axis=1): {t_apply:7.2f} ms")
print(f"vectorized:     {t_vectorized:7.2f} ms")
print(f".apply() was about {t_apply / t_vectorized:.0f}x slower on 10,000 rows.")


Same result as vectorized? True
.apply(axis=1):   37.77 ms
vectorized:        0.30 ms
.apply() was about 127x slower on 10,000 rows.


## Practical Activity — Simulated Diabetes Patient Dataset

A simulated dataset of **500 patients**, one row per patient — feature engineering practice on data that
actually matters.

| Column | What it holds |
|---|---|
| `PTID` | patient ID |
| `Gender`, `Age` | M/F, years |
| `Weight_kg`, `Height_cm` | body measurements |
| `Glucose_mg_dL`, `HbA1c_percent` | blood test results |
| `Blood_Pressure` | **messy**: `"128/82"` (systolic/diastolic together) |
| `Family_Diabetes` | Yes/No, family history |
| `Pregnancy_Count` | 0 for male patients |
| `Target` | 1 = diabetic, 0 = not (the outcome) |

This notebook expects `diabetes_patients.csv` in the same folder as this notebook (it was generated once,
with a fixed random seed, and delivered alongside the slides).


In [67]:
# Load the simulated dataset (see the companion diabetes_patients.csv file)
patients = pd.read_csv("data/diabetes_patients.csv")
patients.head()


,PTID,Gender,Age,Weight_kg,Height_cm,Glucose_mg_dL,HbA1c_percent,Blood_Pressure,Family_Diabetes,Pregnancy_Count,Target
0,P0001,F,90,74.2,167,108,5.4,138/55,Yes,2,0
1,P0002,F,48,90.9,151,85,4.6,141/81,No,1,0
2,P0003,F,38,63.2,161,85,5.2,125/82,No,3,0
3,P0004,M,78,68.1,173,158,7.5,103/67,No,0,0
4,P0005,M,52,113.9,174,71,5.4,112/84,Yes,0,0


### Your tasks

1. **BMI**: derive it from `Weight_kg` and `Height_cm` (careful with the units — height is in cm, the BMI
   formula needs meters!)
2. **Age_Group**: bin `Age` into categories with `pd.cut()`
3. **HbA1c_Category**: bin `HbA1c_percent` using the real clinical cutoffs — Normal < 5.7, Prediabetes
   5.7–6.4, Diabetic ≥ 6.5
4. **Systolic / Diastolic**: split `Blood_Pressure` into two numeric columns
5. **risk_factor_count**: count how many of these are true — high glucose, high HbA1c, family history,
   high BMI

One way to solve it, below.


In [68]:
# --- Task 1: BMI = weight (kg) / height (m)^2 -- note the /100 to convert cm to m ---
patients["BMI"] = (
    patients["Weight_kg"] / (patients["Height_cm"] / 100) ** 2
).round(1)

# --- Task 2: bin Age into groups ---
patients["Age_Group"] = pd.cut(
    patients["Age"], bins=[0, 30, 45, 60, 120],
    labels=["18-30", "31-45", "46-60", "60+"]
)

# --- Task 3: bin HbA1c using real clinical thresholds ---
patients["HbA1c_Category"] = pd.cut(
    patients["HbA1c_percent"], bins=[0, 5.7, 6.5, 20],
    labels=["Normal", "Prediabetes", "Diabetic"]
)

patients[["PTID", "BMI", "Age_Group", "HbA1c_percent", "HbA1c_Category"]].head()


,PTID,BMI,Age_Group,HbA1c_percent,HbA1c_Category
0,P0001,26.6,60+,5.4,Normal
1,P0002,39.9,46-60,4.6,Normal
2,P0003,24.4,31-45,5.2,Normal
3,P0004,22.8,60+,7.5,Diabetic
4,P0005,37.6,46-60,5.4,Normal


In [69]:
# --- Task 4: split the messy "systolic/diastolic" string into two numeric columns ---
bp_split = patients["Blood_Pressure"].str.split("/", expand=True)
patients["Systolic"]  = bp_split[0].astype(int)
patients["Diastolic"] = bp_split[1].astype(int)

# --- Task 5: a composite risk score -- sum of four boolean flags ---
patients["risk_factor_count"] = (
    (patients["Glucose_mg_dL"] > 125).astype(int) +
    (patients["HbA1c_percent"] >= 6.5).astype(int) +
    (patients["Family_Diabetes"] == "Yes").astype(int) +
    (patients["BMI"] >= 30).astype(int)
)

patients[["PTID", "Systolic", "Diastolic", "risk_factor_count"]].head()


,PTID,Systolic,Diastolic,risk_factor_count
0,P0001,138,55,1
1,P0002,141,81,1
2,P0003,125,82,0
3,P0004,103,67,2
4,P0005,112,84,2


### Debrief

Does `risk_factor_count` actually line up with the real outcome (`Target`)? Let's check.


In [70]:
# % of patients who are ACTUALLY diabetic (Target == 1), grouped by how many risk factors they have
debrief = patients.groupby("risk_factor_count")["Target"].agg(
    pct_actually_diabetic=lambda s: f"{s.mean():.0%}",
    patients="count",
)
debrief


,pct_actually_diabetic,patients
risk_factor_count,,
0,9%,110
1,21%,208
2,54%,135
3,81%,42
4,100%,5


Four simple boolean flags, added together, turn into a genuinely predictive feature — this is what
**"feature engineering"** means in Machine Learning. Every one of those flags came from a derived column
you just built. Data Engineering is what makes this possible *before* any model gets trained.


## Recap Quiz — Kahoot!

A quick, competitive recap of everything above: Series & DataFrames, Dates, Indexing & Filtering,
Iteration, Fundamental Operations, Derived Columns. (Question bank kept as a separate document, not part
of this notebook.)


## Hands-On Lab — The Running Project (Olist)

The real running project is **Olist**, a Brazilian e-commerce dataset with multiple linked tables joined
by `order_id` / `customer_id` — real-world mess (missing values, mixed types, dates as text) included on
purpose.

> **Note:** the real `olist_orders_dataset.csv` isn't part of this notebook. So the 5 exercises below run
> end-to-end, we generate a small **synthetic stand-in** with the same columns and the same kind of
> messiness (some canceled orders have no delivery date at all — a realistic missing-value case). Replace
> `orders_sample` below with `pd.read_csv("olist_orders_dataset.csv")` once you have the real file.


### Exercise 1 — Load & Inspect

**Goal:** confirm you can tell, in your own words, what a row represents.


In [77]:
orders = pd.read_csv("data/olist_orders_dataset.csv")
orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Exercise 2 — Dates

**Goal:** answer — which weekday has the most orders?


In [73]:
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_dow"]   = orders["order_purchase_timestamp"].dt.day_name()

orders["purchase_dow"].value_counts()


purchase_dow
Wednesday    59
Thursday     47
Tuesday      47
Friday       45
Saturday     38
Monday       37
Sunday       27
Name: count, dtype: int64

### Exercise 3 — Select & Filter

**Goal:** what fraction of orders were delivered late?


In [74]:
june_orders = orders[orders["purchase_month"] == 6]
delivered   = orders[orders["order_status"] == "delivered"]
late_ones   = orders[
    orders["order_delivered_customer_date"]
    > orders["order_estimated_delivery_date"]
]

print("june orders:   ", june_orders.shape)
print("delivered:     ", delivered.shape)
print("late:          ", late_ones.shape)
print(f"fraction late: {len(late_ones) / len(orders):.1%}")


june orders:    (41, 9)
delivered:      (257, 9)
late:           (62, 9)
fraction late: 20.7%


### Exercise 4 — Describe

**Goal:** write one sentence describing this dataset, as if to a new teammate.


In [75]:
orders["order_status"].value_counts()


order_status
delivered    257
shipped       32
canceled      11
Name: count, dtype: int64

### Exercise 5 — Derive

**Goal:** create at least one derived column of your own that isn't shown here — and be ready to explain
why it's useful.


In [76]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

orders["was_late"] = (
    orders["order_delivered_customer_date"]
    > orders["order_estimated_delivery_date"]
)

orders[["order_id", "delivery_days", "was_late"]].head()


,order_id,delivery_days,was_late
0,ord_1001,14.0,False
1,ord_1002,19.0,True
2,ord_1003,18.0,True
3,ord_1004,4.0,False
4,ord_1005,5.0,False


## Before Next Class

- **Homework:** finish any unfinished lab exercise; save your notebook.
- **Next topic (2.2):** loading & storing data — files, SQL (SQLAlchemy), NoSQL (PyMongo / Firestore).
- **Bring:** questions from today — pandas gets easier once the basics click.

Great first real class — see you next week!

`eduardo.deavila@tec.mx` | Tel. 4925442458
